# 2025-26 Project - Foundations of Computer Science

**Project by:**
- Herandez, Nicole Winy </br>
- Ortega, Azriel Matthew </br>
- Tibayan, Meryann Joy

In [ ]:
# import libraries
import pandas as pd
import os

### 1. Create a single dataframe with the concatenation of all input csv files, adding a column called country

In [ ]:
# read csv files with read_csv function from pandas
videos_folder = "videos"

df_cavideos = pd.read_csv(f"{videos_folder}/CAvideos.csv")
df_devideos = pd.read_csv(f"{videos_folder}/DEvideos.csv")
df_frvideos = pd.read_csv(f"{videos_folder}/FRvideos.csv")
df_gbvideos = pd.read_csv(f"{videos_folder}/GBvideos.csv")
df_invideos = pd.read_csv(f"{videos_folder}/INvideos.csv")
df_usvideos = pd.read_csv(f"{videos_folder}/USvideos.csv")

# TO DO: Experiment with these 4 CSV Files with different encoding to see which has the best result

# UPDATE 1: After experimenting, it seems that using default encoding(utf-8) and using encoding errors
# to skip some bytes that cant be read is better than finding a specific encoding for each language
# However, this must be researched more for example: If this only a computer specific problem, or 
# there is a way to read everything properly without having to use encoding errors

df_jpvideos = pd.read_csv(f"{videos_folder}/JPvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_krvideos = pd.read_csv(f"{videos_folder}/KRvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_mxvideos = pd.read_csv(f"{videos_folder}/MXvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")
df_ruvideos = pd.read_csv(f"{videos_folder}/RUvideos.csv", encoding="utf-8", encoding_errors="replace", engine="python")

They say it is hard to determine the correct encoding for each text file, especially if it is not utf-8 </br>
Reference https://community.dataquest.io/t/how-do-we-know-the-encoding-to-use-with-pd-read-csv-chardet-doesnt-help/467614/4 </br>

You have to do trial and error with all of the encodings to check which one works best. As for my testing 11/20/2025, utf-8 is best for all with the encoding_errors set to replace.

Here is the docs for checking all encodings : https://docs.python.org/3/library/codecs.html#standard-encodings

In [ ]:
# Efficient code version (with concatenation process

# Import all csv files, read and compile them into one dataframe
# With the addition of adding a new column 'country' to determine from what csv file it came from

dataframe_lists = []
for file in os.listdir(videos_folder):
    if file[9:] == "csv":
        print(f"Reading {file}")
        df_temp = pd.read_csv(f"{videos_folder}/{file}", encoding_errors="replace")
        df_temp["country"] = file[:2] # add country column
        dataframe_lists.append(df_temp)

df_videos = pd.concat(dataframe_lists) # concatenate everything into one dataframe
df_videos = df_videos.reset_index(drop=True) # reset index

In [ ]:
df_videos

### 2. Extract all videos that have no tag.

In [ ]:
no_tag_videos = df_videos[
    (df_videos["tags"].isna()) |
    (df_videos["tags"] == "") |
    (df_videos["tags"] == "[none]")
]

no_tag_videos 

### 3. For each channel, determine the total number of views

In [ ]:
total_views_per_channel = df_videos.groupby("channel_title")["views"].sum()

with pd.option_context('display.max_rows', None):
    display(total_views_per_channel)

In [ ]:
total_views_per_channel = df_videos.groupby("channel_title")["views"].sum().reset_index()

with pd.option_context('display.max_rows', None):
    display(total_views_per_channel)

### 4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.

In [ ]:
df_videos_disabled = df_videos[df_videos["comments_disabled"] | df_videos["video_error_or_removed"]]
df_videos = df_videos.drop(df_videos_disabled.index)

### 5. Add a like_ratio column storing the ratio between the number of likes and of dislikes

In [ ]:
df_videos["like_ratio"] = df_videos["likes"] / df_videos["dislikes"].replace(0, 1)
df_videos[["title", "likes", "dislikes", "like_ratio"]].head(10)

### 6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

In [ ]:
df_videos["publish_time"] = pd.to_datetime(df_videos["publish_time"])
#df_videos["publish_10min"] = df_videos["publish_time"].dt.floor("10min")

In [ ]:
df_videos["publish_interval"] = (
    df_videos["publish_time"].dt.floor("10min").dt.strftime("%H:%M") 
    + " - " +
    (df_videos["publish_time"].dt.floor("10min") + pd.Timedelta(minutes=10)).dt.strftime("%H:%M")
)

In [ ]:
with pd.option_context('display.max_rows', None):
    df_videos[["publish_time", "publish_interval"]].head(10)

Abbiamo trasformato l’orario di pubblicazione di ogni video in un intervallo di 10 minuti.  
Questo ci permette di raggruppare i video in fasce orarie più ampie e più facili da analizzare.

Per prima cosa abbiamo convertito la colonna `publish_time` in un formato datetime, per essere sicuri che Python la riconosca come un orario.

Successivamente abbiamo usato la funzione `dt.floor("10min")` per “arrotondare verso il basso” l’orario al suo intervallo di 10 minuti.  

Infine abbiamo creato una colonna opzionale `publish_interval` per mostrare l’intervallo in modo più chiaro, ad esempio `"06:20 - 06:30"`.  
Questo rende l’analisi dei tempi di pubblicazione più intuitiva e immediata.

### 7. For each interval, determine the number of videos, average number of likes and of dislikes.

In [ ]:
interval_stats = df_videos.groupby("publish_interval").agg(
    number_of_videos = ("video_id", "count"),
    average_likes = ("likes", "mean"),
    average_dislikes = ("dislikes", "mean")
)

interval_stats.head(50)

### 8. For each tag, determine the number of videos. Notice that tags contains a string with several tags

In [ ]:
# Setup tag column in our dataframe first
df_videos["tags"] = df_videos["tags"].str.split('|') # Split the string into array with '|' as the separator

# Explode dataframe so that each row wiill have one tag, it will duplicate rows with many tags
# This is the approach because it is useful when grouping by tags.
df_videos_tags_explode = df_videos.explode('tags') 

In [ ]:
# clean dataset by removing all extra characters such as double quotations (")
df_videos_tags_explode["tags"] = df_videos_tags_explode["tags"].str.replace('"', '')

In [ ]:
df_videos_tags_explode["tags"].value_counts()

### 9. Find the tags with the largest number of videos

In [ ]:
df_videos_tags_explode.groupby(by="tags")["views"].sum() 

### 10. For each (tag, country) pair, compute average ratio likes/dislikes

In [ ]:
pd.DataFrame(df_videos_tags_explode.groupby(by=["tags","country"])["like_ratio"].mean())

### 11. For each (trending_date, country) pair, the video with the largest number of views

In [ ]:
df_videos.columns

In [ ]:
df_videos_group = df_videos.groupby(by=["trending_date", "country"])["views"].idxmax()
df_videos_group = df_videos.loc[df_videos_group]
df_videos_group

In [ ]:
# Sort columns, to display Country, Trending Date and then Views First.
priority_columns = ["trending_date","country", "views"]
other_columns = [col for col in df_videos.columns if col not in priority_columns]

df_videos_group[priority_columns + other_columns]

### 12. Divide trending_date into three columns: year, month, day

In [ ]:
# 1. Suddividi la colonna 'trending_date'
# Il parametro expand=True trasforma il risultato in un DataFrame
data_split = df_videos['trending_date'].str.split('.', expand=True)

# 2. Assegna le nuove colonne
# Indice 0 = Anno (YY) -> aggiungiamo '20' per avere '2017'
df_videos['year'] = '20' + data_split[0].astype(str)
# Indice 1 = Giorno (DD)
df_videos['day'] = data_split[1]
# Indice 2 = Mese (MM)
df_videos['month'] = data_split[2]

# Verifica il risultato
print(df_videos[['video_id','trending_date', 'year', 'month', 'day']].head())

In [ ]:
# 1. Conversione in formato datetime
# Utilizza il formato corretto che sembra essere 'YY.DD.MM'
df_videos['trending_date'] = pd.to_datetime(df_videos['trending_date'], format='%y.%d.%m')

# 2. Estrazione delle componenti
df_videos['year'] = df_videos['trending_date'].dt.year
df_videos['month'] = df_videos['trending_date'].dt.month
df_videos['day'] = df_videos['trending_date'].dt.day

# Stampa le prime righe per verificare il risultato
print(df_videos[['trending_date', 'year', 'month', 'day']].head())

### 13. For each (month, country) pair, the video with the largest number of views

In [ ]:
# 1. Raggruppa per 'month' e 'country'
# 2. Trova l'indice della riga con il valore 'views' più alto in ogni gruppo
max_views_indices = df_videos.groupby(['month', 'country'])['views'].idxmax()

# 3. Usa gli indici trovati per selezionare le righe complete dal DataFrame originale
df_max_views = df_videos.loc[max_views_indices]

# Visualizza il risultato
print(df_max_views[['month', 'country', 'video_id','title', 'views']].head())


### 14. Read all json files with the video categories

In [ ]:
json_folder = 'categories'
df_list = []

for file in os.listdir(json_folder):
    if file.endswith(".json"):
        print(f"Reading {file}")
        
        with open(os.path.join(json_folder, file), 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Normalizzazione corretta per YouTube categories
        df_temp = pd.json_normalize(data['items'])
        
        # Selezioniamo solo le colonne utili
        df_temp = df_temp[['id', 'snippet.title', 'snippet.assignable']]
        
        # Rinominiamo per coerenza con dataframe video
        df_temp.columns = ['category_id', 'category_name', 'assignable']
        
        # Convertiamo category_id in numero intero
        df_temp['category_id'] = df_temp['category_id'].astype(int)
        
        df_list.append(df_temp)

df_categories = pd.concat(df_list, ignore_index=True)
df_categories = df_categories.drop_duplicates(subset=['category_id']).reset_index(drop=True)

df_categories


### 15. For each country, determine how many videos have a category that is not assignable

In [ ]:
df_merged = df_videos.merge(df_categories, on='category_id', how='left')
df_not_assignable = (df_merged[df_merged['assignable'] == False]
    .groupby('country')['video_id']
    .count()
    .reset_index(name='non_assignable_count')
)
print(df_not_assignable)
